# RETFound Training Notebook
Notebook per il training del modello con checkpoint nominati in base agli hyperparametri.

In [ ]:
! git clone -b 'dropout' "https://github.com/fraco03/DR_RETFound.git"
%cd "/kaggle/working/DR_RETFound"
! pwd


In [ ]:
!pip install grad-cam
import os
import pandas as pd
import numpy as np
import torch
import wandb
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import ConcatDataset, DataLoader, WeightedRandomSampler

from src.dataset import RetinopathyDataset, get_transforms
from src.utils import visualize_attention_retfound
from src.loss import SmoothL1Loss
from src.model_setup import build_retfound_regression

In [ ]:
def build_config():
    return {
        'aptos_csv': '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv',
        'messidor_csv': '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv',
        'aptos_img_dir': '/kaggle/input/competitions/aptos2019-blindness-detection/train_images',
        'messidor_img_dir': '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess',
        'weights_path': '/kaggle/input/models/fraco0344/retfound/pytorch/default/1/RETFound_mae_natureCFP.pth',
        'checkpoint_dir': 'weights',
        'test_size': 0.2,
        'random_state': 42,
        'batch_size': 32,
        'num_workers': 4,
        'epochs': 20,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'loss_beta': 1.0,
        'sampler': 'weighted',
        'num_classes': 5,
        'log_confusion_matrix': True,
        'use_wandb': True,
        'wandb_project': 'dr-retfound',
        'wandb_run_name': None,
        'patience': 2,
        'early_stop_patience': 5,
        'clahe': True,
        'drop_path_rate': 0.1,
    }

cfg = build_config()
cfg

In [ ]:
def build_run_name(cfg):
    parts = [
        f"bs{cfg['batch_size']}",
        f"lr{cfg['lr']}",
        f"wd{cfg['weight_decay']}",
        f"ep{cfg['epochs']}",
        f"beta{cfg['loss_beta']}",
        f"sampler{cfg['sampler']}",
        f"seed{cfg['random_state']}",
        f"clahe{cfg['clahe']}",
    ]
    return "retfound__" + "__".join(parts)

if cfg['use_wandb']:
    # Removed try-except to expose the real error
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    
    # This will crash if the secret doesn't exist or isn't attached
    wandb_key = user_secrets.get_secret("WANDB")
    
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ["WANDB_MODE"] = "online"
    
    # Explicitly use the 'key=' argument
    wandb.login(key=wandb_key)
    
    run_name = cfg['wandb_run_name'] or build_run_name(cfg)
    wandb.init(project=cfg['wandb_project'], name=run_name, config=cfg)

In [ ]:
def load_splits(cfg):
    df_messidor = pd.read_csv(cfg['messidor_csv'])
    df_aptos = pd.read_csv(cfg['aptos_csv'])

    aptos_train, aptos_val = train_test_split(
        df_aptos,
        test_size=cfg['test_size'],
        stratify=df_aptos['diagnosis'],
        random_state=cfg['random_state'],
    )

    messidor_train, messidor_val = train_test_split(
        df_messidor,
        test_size=cfg['test_size'],
        stratify=df_messidor['diagnosis'],
        random_state=cfg['random_state'],
    )

    return aptos_train, aptos_val, messidor_train, messidor_val

def build_datasets(cfg):
    aptos_train, aptos_val, messidor_train, messidor_val = load_splits(cfg)

    train_transform = get_transforms(is_train=True, clahe=cfg['clahe'])
    val_transform = get_transforms(is_train=False, clahe=cfg['clahe'])

    ds_aptos_train = RetinopathyDataset(
        aptos_train,
        cfg['aptos_img_dir'],
        transform=train_transform,
    )
    ds_messidor_train = RetinopathyDataset(
        messidor_train,
        cfg['messidor_img_dir'],
        transform=train_transform,
    )

    ds_aptos_val = RetinopathyDataset(
        aptos_val,
        cfg['aptos_img_dir'],
        transform=val_transform,
    )
    ds_messidor_val = RetinopathyDataset(
        messidor_val,
        cfg['messidor_img_dir'],
        transform=val_transform,
    )

    train_dataset = ConcatDataset([ds_messidor_train, ds_aptos_train])
    val_dataset = ConcatDataset([ds_messidor_val, ds_aptos_val])

    return train_dataset, val_dataset

train_dataset, val_dataset = build_datasets(cfg)
len(train_dataset), len(val_dataset)

In [ ]:
def _extract_labels_from_dataset(ds):
    if isinstance(ds, RetinopathyDataset):
        return ds.df.iloc[:, 1].to_numpy(dtype=np.float32)
    if hasattr(ds, "df"):
        return ds.df.iloc[:, 1].to_numpy(dtype=np.float32)
    return None

def build_sampler(train_dataset, batch_size=512, num_workers=0):
    labels = None
    if isinstance(train_dataset, ConcatDataset):
        parts = []
        for ds in train_dataset.datasets:
            part = _extract_labels_from_dataset(ds)
            if part is None:
                parts = None
                break
            parts.append(part)
        if parts is not None:
            labels = np.concatenate(parts)
    else:
        labels = _extract_labels_from_dataset(train_dataset)

    if labels is None:
        label_batches = []
        loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
        )
        for _, batch_labels in loader:
            label_batches.append(batch_labels.view(-1).cpu())
        labels = torch.cat(label_batches).numpy()

    class_counts = np.bincount(np.round(labels).astype(int))
    print(f"Distribuzione immagini per classe: {class_counts}")

    class_weights = 1.0 / (class_counts + 1e-5)
    sample_weights = class_weights[np.round(labels).astype(int)]
    sample_weights = torch.from_numpy(sample_weights).float()

    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

def build_loaders(cfg, train_dataset, val_dataset):
    train_sampler = None
    if cfg['sampler'] == 'weighted':
        train_sampler = build_sampler(
            train_dataset,
            batch_size=cfg['batch_size'] * 4,
            num_workers=cfg['num_workers'],
        )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg['batch_size'],
        sampler=train_sampler,
        shuffle=train_sampler is None,
        num_workers=cfg['num_workers'],
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=cfg['num_workers'],
    )

    return train_loader, val_loader

train_loader, val_loader = build_loaders(cfg, train_dataset, val_dataset)
len(train_loader), len(val_loader)

In [ ]:
def save_checkpoint(cfg, model, optimizer, epoch_idx, val_loss):
    os.makedirs(cfg['checkpoint_dir'], exist_ok=True)
    ckpt_path = os.path.join(cfg['checkpoint_dir'], 'retfound.pt')

    torch.save(
        {
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'epoch': epoch_idx,
            'val_loss': val_loss,
            'config': cfg,
        },
        ckpt_path,
    )
    return ckpt_path

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images, labels = images.to(device), labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images).squeeze()
        batch_loss = loss_fn(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        total_loss += batch_loss.item()
    return total_loss / max(1, len(loader))

from sklearn.metrics import confusion_matrix, cohen_kappa_score
import numpy as np
import torch
from tqdm import tqdm

def eval_one_epoch(model, loader, loss_fn, device, num_classes, compute_cm=False):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val', leave=False):
            images, labels = images.to(device), labels.float().to(device)
            outputs = model(images).squeeze()
            batch_loss = loss_fn(outputs, labels)
            total_loss += batch_loss.item()
            
            if compute_cm:
                # Clamp predictions and targets to valid class ranges and convert to int
                preds = outputs.round().clamp(0, num_classes - 1).long().cpu().numpy()
                targets = labels.round().clamp(0, num_classes - 1).long().cpu().numpy()
                all_preds.append(preds)
                all_targets.append(targets)
                
    avg_loss = total_loss / max(1, len(loader))
    
    if compute_cm:
        all_preds = np.concatenate(all_preds) if all_preds else np.array([])
        all_targets = np.concatenate(all_targets) if all_targets else np.array([])
        
        # Calculate standard confusion matrix
        cm = confusion_matrix(all_targets, all_preds, labels=list(range(num_classes)))
        
        # Calculate Quadratic Weighted Kappa
        kappa = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
        
        return avg_loss, cm, kappa
        
    return avg_loss, None, None

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_retfound_regression(cfg['weights_path'], device=device, drop_path_rate=cfg['drop_path_rate'])
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
loss_fn = SmoothL1Loss(beta=cfg['loss_beta'])
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg['lr'],
    weight_decay=cfg['weight_decay'],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # Vogliamo minimizzare la loss (se usi un'accuratezza/kappa, metteresti 'max')
    factor=0.5,       # Dimezza il learning rate quando interviene (es. da 1e-4 a 5e-5)
    patience=cfg['patience'],       # Aspetta 2 epoche senza miglioramenti prima di agire
    min_lr=1e-7,      # Limite di sicurezza sotto il quale il LR non scenderà mai
)

In [ ]:
cm_display = None
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch_idx in range(1, cfg['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, cm, kappa = eval_one_epoch(
        model,
        val_loader,
        loss_fn,
        device,
        num_classes=cfg['num_classes'],
        compute_cm=cfg['log_confusion_matrix'],
    )
    scheduler.step(val_loss)
    print(
        f"Epoch {epoch_idx}/{cfg['epochs']}, "
        f"Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}, Kappa: {kappa:.4f}"
    )

    if cfg['log_confusion_matrix'] and cm is not None:
        disp = ConfusionMatrixDisplay(cm, display_labels=list(range(cfg['num_classes'])))
        fig, ax = plt.subplots(figsize=(5, 5))
        disp.plot(ax=ax, colorbar=False)
        ax.set_title(f"Val Confusion Matrix - Epoch {epoch_idx}")
        if cm_display is None:
            cm_display = display(fig, display_id=True)
        else:
            cm_display.update(fig)
        if cfg['use_wandb']:
            wandb.log({"val_confusion_matrix": wandb.Image(fig), "epoch": epoch_idx})
        plt.close(fig)

    if cfg['use_wandb']:
        wandb.log({
            'epoch': epoch_idx,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'kappa': kappa
        })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        ckpt_path = save_checkpoint(cfg, model, optimizer, epoch_idx, val_loss)
        print(f"Checkpoint salvato: {ckpt_path}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg['early_stop_patience']:
            print("Early stopping: nessun miglioramento della validation loss.")
            break

if cfg['use_wandb']:
    wandb.finish()

In [ ]:
import numpy as np
import torch
import random

# 1. Peschiamo un indice a caso da tutto il Validation Dataset
random_idx = random.randint(0, len(val_dataset) - 1)

# Estraiamo l'immagine e l'etichetta direttamente dal dataset
image_tensor, label = val_dataset[random_idx]

# Il dataset ci restituisce [3, 224, 224], ma al modello serve la dimensione del batch.
# Usiamo unsqueeze per farla diventare [1, 3, 224, 224]
image_tensor = image_tensor.unsqueeze(0)
vero_grado = label.item() if isinstance(label, torch.Tensor) else float(label)

# 2. Predizione istantanea 
model.eval()
with torch.no_grad():
    predizione = model(image_tensor.to(device)).item()

print(f"Paziente n° {random_idx} | Grado Reale: {vero_grado} | Predizione Modello: {predizione:.2f}")

# 3. Prepariamo l'immagine per i nostri occhi (S-Normalizzazione)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

img_vis = image_tensor.squeeze().cpu().numpy().transpose(1, 2, 0)
img_vis = std * img_vis + mean
img_vis = np.clip(img_vis, 0, 1)

# 4. Magia!
visualizza_attenzione_retfound(model, image_tensor, img_vis, device)